# CAS Exam 5: Recoveries and Reinsurance in Reserving

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, Casualty Actuarial Society, 2010 — Chapters 17–18

**Exam tasks covered:** B13 (Recoveries: salvage, subrogation, reinsurance), B19 (Reinsurance concepts: net/ceded/gross)

**Learning goals:**
1. Define and distinguish salvage, subrogation, and reinsurance recoveries
2. Estimate salvage and subrogation recoveries using the percentage-of-losses and development methods
3. Define gross, ceded, and net losses in the context of a reserving analysis
4. Apply quota share and excess-of-loss treaty terms to development triangles
5. Understand the two approaches to projecting net-of-reinsurance reserves

## Formula Sheet Quick Reference

### Salvage and Subrogation

| Formula | Notes |
|---|---|
| $\text{Net Paid Loss} = \text{Gross Paid Loss} - \text{Recovery}$ | Applied cell-by-cell if developing net |
| $\text{Recovery} = r \times \text{Ultimate Gross Loss}$ | Ratio method: select ratio $r$ from history |
| $\text{Recovery IBNR} = \text{Recovery Ultimate} - \text{Recovery Paid to Date}$ | Treat recovery triangle like a loss triangle |

### Reinsurance

| Concept | Formula |
|---|---|
| **Gross losses** | Direct losses before any reinsurance cession |
| **Ceded losses** | Amount transferred to reinsurer under treaty |
| **Net losses** | $\text{Net} = \text{Gross} - \text{Ceded}$ |
| **Quota share ceded** | $\text{Ceded} = QS\% \times \text{Gross}$ |
| **Per-occurrence XOL ceded** | $\text{Ceded per occurrence} = \min(\text{Gross}, \text{Limit}) - \min(\text{Gross}, \text{Retention})$ |
| **Net IBNR** | $\text{Net IBNR} = \text{Net Ultimate} - \text{Net Latest}$ |
| **Ceded IBNR** | $\text{Ceded IBNR} = \text{Ceded Ultimate} - \text{Ceded Latest}$ |

## Salvage and Subrogation

### Definitions

**Salvage** — the amount an insurer recovers from the sale or use of property that has been declared a total loss. After paying a total loss claim (e.g., a totaled vehicle), the insurer takes title to the damaged property and sells it to a salvage dealer or at auction. The sale proceeds reduce the insurer's net cost.

*Example:* An auto insurer pays $18,000 for a totaled vehicle. The insurer then sells the damaged vehicle for $3,500 at a salvage auction. Net cost = $18,000 − $3,500 = $14,500.

**Subrogation** — the insurer's legal right to recover loss payments from the party responsible for causing the loss, after paying the policyholder's claim. The insurer "steps into the shoes" of the policyholder and pursues the at-fault party or their insurer.

*Example:* An insurer pays $25,000 to a policyholder whose vehicle was totaled in an accident caused by another driver. The insurer then sues the at-fault driver's liability insurer and recovers $22,000. Net cost = $25,000 − $22,000 = $3,000.

### Key Properties

| Property | Salvage | Subrogation |
|---|---|---|
| **Nature** | Property recovery | Third-party legal recovery |
| **Common lines** | Auto physical damage, property | Auto liability, GL, workers comp |
| **Timing** | Usually faster (auction within weeks) | Often slower (litigation can take years) |
| **Certainty** | More certain (market price) | More uncertain (depends on collectability) |
| **Accounting** | Reduces losses when received | Reduces losses when received |

**Both recoveries reduce net incurred losses.** In statutory accounting, they are recorded as negative losses when received, reducing the paid loss total.

> **Exam tip** — Salvage and subrogation are RECOVERIES, not negative claims. The distinction matters for data organization: recoveries are added to the triangle as negative values after they are received. Anticipating future recoveries creates an IBNR-equivalent obligation for recovery estimation.

---

### Two Approaches to Estimating Net Losses

**Approach 1 — Develop net losses directly:**
- The historical loss triangle is already net of all salvage and subrogation
- Apply standard chain ladder or other development methods to the net triangle
- Simple; appropriate if recovery patterns are stable and data is consistently net
- Problem: if recoveries are large and variable, the net triangle may be volatile and hard to develop

**Approach 2 — Develop gross, then project recoveries separately:**
- Step 1: Develop gross losses to ultimate using standard methods
- Step 2: Estimate future salvage/subrogation (see methods below)
- Step 3: Net ultimate = Gross ultimate − Recovery ultimate
- More transparent; shows the gross and recovery components separately
- Required when historical data is gross only (net data not maintained separately)

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

# ─── Salvage Estimation: Percentage of Paid Losses Method ─────────────────────
# Historical paid gross losses and salvage received by accident year (as of latest)

data = {
    'AY': [2019, 2020, 2021, 2022, 2023],
    'Gross_Ultimate': [10_200, 11_500, 12_100, 12_800, 13_400],  # fully developed gross
    'Salvage_Received': [  680,    740,    790,    830,    510],   # 2023 is immature
}
df = pd.DataFrame(data).set_index('AY')

# Historical recovery ratio for mature years only
df['Recovery_Ratio'] = df['Salvage_Received'] / df['Gross_Ultimate']

# For 2023, the triangle is immature — exclude from ratio calculation
mature_mask = [2019, 2020, 2021, 2022]
historical_ratios = df.loc[mature_mask, 'Recovery_Ratio']
selected_ratio = historical_ratios.mean()
print(f'Historical salvage-to-gross-loss ratios (mature AYs):')
print(historical_ratios.map('{:.3f}'.format).to_frame('ratio'))
print(f'\nSelected recovery ratio: {selected_ratio:.3f}')

In [ ]:
# ─── Apply selected ratio to project ultimate recoveries ──────────────────────
df['Projected_Recovery_Ult'] = df['Gross_Ultimate'] * selected_ratio
df['Recovery_IBNR'] = df['Projected_Recovery_Ult'] - df['Salvage_Received']
df['Net_Ultimate'] = df['Gross_Ultimate'] - df['Projected_Recovery_Ult']

print('Salvage projection — percentage-of-losses method:')
print(df[['Gross_Ultimate', 'Salvage_Received', 'Projected_Recovery_Ult',
          'Recovery_IBNR', 'Net_Ultimate']].to_string())
print(f'\nTotal Recovery IBNR: {df["Recovery_IBNR"].sum():,.0f}')
print(f'Total Net Ultimate:  {df["Net_Ultimate"].sum():,.0f}')
print()
print('Interpretation: the recovery IBNR is the amount of additional salvage')
print('expected to be received in future periods on already-paid gross losses.')

In [ ]:
# ─── Development Method on Recovery Triangle ──────────────────────────────────
# If the insurer maintains a triangle of cumulative salvage received by AY × age,
# it can be developed like a loss triangle.

# Hypothetical recovery triangle (cumulative salvage received, $000s)
# Columns: 12 months, 24 months, 36 months, 48 months
recovery_triangle = pd.DataFrame({
    '12':  [240,  270,  280,  285,  np.nan],
    '24':  [520,  590,  605,  np.nan, np.nan],
    '36':  [640,  720,  np.nan, np.nan, np.nan],
    '48':  [680,  np.nan, np.nan, np.nan, np.nan],
}, index=[2019, 2020, 2021, 2022, 2023])

print('Cumulative recovery triangle ($000s):')
print(recovery_triangle.to_string())

# Compute link ratios
ldfs = {}
ages = ['12', '24', '36', '48']
for i in range(len(ages) - 1):
    from_age, to_age = ages[i], ages[i+1]
    mask = recovery_triangle[[from_age, to_age]].notna().all(axis=1)
    num = recovery_triangle.loc[mask, to_age].sum()
    den = recovery_triangle.loc[mask, from_age].sum()
    ldfs[f'{from_age}-{to_age}'] = num / den

print('\nVolume-weighted link ratios for recovery triangle:')
pd.Series(ldfs).map('{:.4f}'.format).to_frame('LDF')

In [ ]:
# Develop recovery triangle to ultimate using computed LDFs + tail = 1.00
# (assume recoveries fully realized by 48 months)

latest_recovery = {
    2019: (recovery_triangle.loc[2019, '48'], '48'),
    2020: (recovery_triangle.loc[2020, '36'], '36'),
    2021: (recovery_triangle.loc[2021, '24'], '24'),
    2022: (recovery_triangle.loc[2022, '12'], '12'),
    2023: (0, '0'),  # no recoveries yet for most recent year
}

# Build CDF from each age to 48 (then × 1.0 tail)
cdf_to_ult = {
    '48': 1.000,
    '36': ldfs['36-48'] * 1.000,
    '24': ldfs['24-36'] * ldfs['36-48'] * 1.000,
    '12': ldfs['12-24'] * ldfs['24-36'] * ldfs['36-48'] * 1.000,
    '0':  ldfs['12-24'] * ldfs['24-36'] * ldfs['36-48'] * 1.000 * 1.25,  # assume 25% more for 2023
}

rows = []
for ay, (latest, age) in latest_recovery.items():
    cdf = cdf_to_ult.get(age, 1.0)
    ult = latest * cdf
    rows.append({'AY': ay, 'Latest Recovery': latest, 'Age': age, 'CDF': cdf, 'Ult Recovery': ult})

result = pd.DataFrame(rows).set_index('AY')
print('Recovery triangle development to ultimate:')
print(result.to_string())
print(f'\nTotal projected recovery ultimate: {result["Ult Recovery"].sum():,.0f}')

## Reinsurance in Reserving

### Gross / Ceded / Net Hierarchy

Every loss that an insurer incurs can be decomposed into three layers:

$$\text{Gross Loss} = \text{Ceded Loss} + \text{Net Loss}$$

| Term | Definition | Also Called |
|---|---|---|
| **Gross losses** | Total direct losses before any reinsurance | Direct losses, before-reinsurance |
| **Ceded losses** | Portion transferred to the reinsurer under treaty or facultative contract | Reinsurer's share |
| **Net losses** | What the primary insurer actually pays after reinsurance recoveries | Retained losses, after-reinsurance |

The same hierarchy applies to IBNR:

$$\text{Gross IBNR} = \text{Ceded IBNR} + \text{Net IBNR}$$

> **Key insight** — Gross IBNR is NOT simply (1 − QS%) of net IBNR for excess-of-loss treaties. The reinsurer's share of IBNR is **not proportional** to their share of paid losses because large claims (which are ceded) develop differently from small claims. This is the central challenge of reinsurance reserving.

---

### Types of Reinsurance Relevant to Reserving

**Quota Share (Proportional)**
- Reinsurer assumes a fixed percentage $q$ of every loss and receives the same percentage of premium
- $\text{Ceded} = q \times \text{Gross}$ for each and every claim
- $\text{Net} = (1-q) \times \text{Gross}$ for each and every claim
- Development patterns of net and gross are **identical** under a pure quota share
- This means: $\text{Net IBNR} = (1-q) \times \text{Gross IBNR}$

**Per-Occurrence Excess of Loss (XOL)**
- Reinsurer pays losses above a retention $R$ per occurrence, up to a limit $L$ per occurrence
- For a single occurrence: $\text{Ceded} = \max(0,\, \min(\text{Gross}, R+L) - R)$
- Equivalently: $\text{Ceded} = \min(\text{Gross}, R+L) - \min(\text{Gross}, R)$
- Net = $\min(\text{Gross},\, R)$ (capped at retention)
- The excess layer develops differently from the primary layer:
  - Small claims (< $R$) are entirely net; they develop like primary losses
  - Large claims (> $R$) are partially or fully ceded; they continue developing in the excess layer for much longer

**Aggregate Excess of Loss (Stop-Loss)**
- Reinsurer pays aggregate losses for an accident year above a retention after all per-occurrence losses are accumulated
- Less common in practice; usually used for catastrophe protection
- Ceded amounts are not known until the accident year is nearly fully developed

In [ ]:
# ─── Quota Share Example ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

# 25% quota share treaty
qs_pct = 0.25

# Gross ultimates and latest paid by AY
qs_data = pd.DataFrame({
    'Gross_Ultimate': [8_500, 9_200, 10_100, 11_400, 12_600],
    'Gross_Latest':   [8_500, 9_200,  8_900,  7_200,  3_100],
}, index=[2019, 2020, 2021, 2022, 2023])

qs_data['Gross_IBNR']    = qs_data['Gross_Ultimate'] - qs_data['Gross_Latest']
qs_data['Ceded_Ultimate'] = qs_data['Gross_Ultimate'] * qs_pct
qs_data['Ceded_Latest']  = qs_data['Gross_Latest'] * qs_pct
qs_data['Ceded_IBNR']    = qs_data['Gross_IBNR'] * qs_pct
qs_data['Net_Ultimate']  = qs_data['Gross_Ultimate'] * (1 - qs_pct)
qs_data['Net_Latest']    = qs_data['Gross_Latest'] * (1 - qs_pct)
qs_data['Net_IBNR']      = qs_data['Gross_IBNR'] * (1 - qs_pct)

print(f'Quota Share Treaty: Reinsurer assumes {qs_pct:.0%} of all losses')
print()
print(qs_data.to_string())
print()
print(f'Total Gross IBNR:  {qs_data["Gross_IBNR"].sum():>10,.0f}')
print(f'Total Ceded IBNR:  {qs_data["Ceded_IBNR"].sum():>10,.0f}  ({qs_pct:.0%} of gross)')
print(f'Total Net IBNR:    {qs_data["Net_IBNR"].sum():>10,.0f}  ({1-qs_pct:.0%} of gross)')
print()
print('Note: Under a pure quota share, ceded IBNR = QS% × gross IBNR exactly.')

In [ ]:
# ─── Excess-of-Loss Example: Applying Treaty Terms to Individual Claims ────────
# Per-occurrence XOL: retention = $500K, limit = $1,000K (i.e., pays $500K xs $500K)

retention = 500_000
limit = 1_000_000

# Individual gross claims by occurrence
claims = pd.DataFrame({
    'Occurrence': ['A', 'B', 'C', 'D', 'E', 'F'],
    'Gross_Loss': [120_000, 380_000, 520_000, 800_000, 1_200_000, 2_000_000],
})

# Apply XOL formula: ceded = min(gross, R+L) - min(gross, R)
claims['Ceded_Loss'] = (
    np.minimum(claims['Gross_Loss'], retention + limit)
    - np.minimum(claims['Gross_Loss'], retention)
)
claims['Net_Loss'] = claims['Gross_Loss'] - claims['Ceded_Loss']
claims['Check_Net'] = np.minimum(claims['Gross_Loss'], retention)  # net = min(gross, retention)

print(f'XOL Treaty: Reinsurer pays {limit/1000:.0f}K excess of {retention/1000:.0f}K per occurrence')
print(f'(Reinsurer covers losses between ${retention/1000:.0f}K and ${(retention+limit)/1000:.0f}K)')
print()
pd.set_option('display.float_format', lambda x: f'${x:,.0f}')
print(claims[['Occurrence', 'Gross_Loss', 'Ceded_Loss', 'Net_Loss']].to_string(index=False))
print()
print(f'Totals:   Gross={claims["Gross_Loss"].sum():>10,.0f}  '
      f'Ceded={claims["Ceded_Loss"].sum():>10,.0f}  '
      f'Net={claims["Net_Loss"].sum():>10,.0f}')
ceded_pct = claims['Ceded_Loss'].sum() / claims['Gross_Loss'].sum()
print(f'\nCeded % of total gross: {ceded_pct:.1%}')
print('Note: the ceded % varies with the severity distribution — it is NOT fixed like QS.')

## Two Approaches to Projecting Net IBNR

### Approach 1 — Develop Net Triangle Directly

**When to use:** The insurer has maintained historical loss triangles on a net-of-reinsurance basis consistently over time.

**Process:**
1. Build the net triangle (paid or incurred losses net of all ceded amounts)
2. Apply standard development methods (chain ladder, BF, etc.) to the net triangle
3. Net ultimate = result directly from the method
4. Net IBNR = Net ultimate − Net latest

**Advantages:**
- Simple and direct
- Automatically reflects the net development pattern
- No need to model reinsurance terms separately

**Disadvantages:**
- Requires consistent historical net data (treaty terms may have changed)
- Cannot separately analyze the ceded portion
- If the treaty structure changed (e.g., retention increased), older rows of the net triangle are not comparable to newer rows

---

### Approach 2 — Gross-minus-Ceded

**When to use:** Treaty terms have changed over time; historical data is only available on a gross basis; or the insurer wants to separately show the ceded IBNR obligation.

**Process:**
1. Develop the **gross triangle** to gross ultimate using standard methods
2. Project **ceded ultimate** by applying current treaty terms to the gross ultimate
   - Quota share: Ceded ultimate = QS% × Gross ultimate
   - XOL per occurrence: more complex — requires assumptions about the severity distribution or use of excess development factors
3. Net ultimate = Gross ultimate − Ceded ultimate
4. Net IBNR = Net ultimate − Net latest
5. Ceded IBNR = Ceded ultimate − Ceded latest

**Why gross IBNR ≠ net IBNR + ceded IBNR separately computed:**

For quota share, the arithmetic works perfectly: Gross IBNR = Net IBNR + Ceded IBNR.

For **excess-of-loss**, the ceded triangle develops more slowly than the gross at early ages (because small losses — which dominate early development — are net), and more heavily at later ages (as large claims finally settle in the excess layer). This means:
- Net development factors < Gross development factors at early ages
- Ceded development factors > Gross development factors at early ages

> **Exam tip** — For XOL treaties, do NOT simply apply (QS%) to gross IBNR to get ceded IBNR. The leverage of reinsurance means the ceded portion of IBNR is disproportionately large relative to the ceded portion of paid losses. The exam frequently tests whether candidates understand this non-proportionality.

---

### The Leveraged Effect of XOL on Development Patterns

| Development Stage | Gross Losses | Net Losses | Ceded Losses |
|---|---|---|---|
| Early ages (immature) | Include many small claims settling quickly | ≈ Gross (small claims are all net) | Small; mostly zero |
| Later ages (mature) | Large claims still developing in excess | Still developing, but slowly | Disproportionately large development |

**Practical consequence:** An insurer with a 25% XOL treaty may have ceded 25% of paid-to-date losses but 40-50% of IBNR if there are large open claims in the excess layer.

In [ ]:
# ─── Gross-minus-Ceded: Quota Share Full Worked Example ───────────────────────
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

# Gross cumulative paid triangle ($000s)
gross_triangle = pd.DataFrame({
    '12':  [3_200, 3_500, 3_800, 4_100, 1_200],
    '24':  [5_800, 6_200, 6_700, np.nan, np.nan],
    '36':  [7_100, 7_600, np.nan, np.nan, np.nan],
    '48':  [7_800, np.nan, np.nan, np.nan, np.nan],
}, index=[2019, 2020, 2021, 2022, 2023])

# Compute gross LDFs (volume-weighted)
ages = ['12', '24', '36', '48']
gross_ldfs = {}
for i in range(len(ages) - 1):
    f, t = ages[i], ages[i+1]
    mask = gross_triangle[[f, t]].notna().all(axis=1)
    gross_ldfs[f'{f}-{t}'] = gross_triangle.loc[mask, t].sum() / gross_triangle.loc[mask, f].sum()

# Tail = 1.00 (assume fully developed at 48 months)
cdf_from_age = {
    '48': 1.0,
    '36': gross_ldfs['36-48'],
    '24': gross_ldfs['24-36'] * gross_ldfs['36-48'],
    '12': gross_ldfs['12-24'] * gross_ldfs['24-36'] * gross_ldfs['36-48'],
}

print('Gross LDFs (volume-weighted):')
pd.Series(gross_ldfs).map('{:.4f}'.format).to_frame('LDF')

In [ ]:
# ─── Apply QS treaty to project net reserves ─────────────────────────────────
qs = 0.30  # 30% quota share

# Latest gross values and development ages
latest_info = {
    2019: (gross_triangle.loc[2019, '48'], '48'),
    2020: (gross_triangle.loc[2020, '36'], '36'),
    2021: (gross_triangle.loc[2021, '24'], '24'),
    2022: (gross_triangle.loc[2022, '12'], '12'),
    2023: (gross_triangle.loc[2023, '12'], '12'),
}

rows = []
for ay, (gross_latest, age) in latest_info.items():
    cdf = cdf_from_age[age]
    gross_ult = gross_latest * cdf
    gross_ibnr = gross_ult - gross_latest

    ceded_ult = gross_ult * qs
    ceded_latest = gross_latest * qs
    ceded_ibnr = ceded_ult - ceded_latest

    net_ult = gross_ult * (1 - qs)
    net_latest = gross_latest * (1 - qs)
    net_ibnr = net_ult - net_latest

    rows.append({
        'AY': ay,
        'Gross Ult': gross_ult, 'Gross Latest': gross_latest, 'Gross IBNR': gross_ibnr,
        'Ceded Ult': ceded_ult, 'Ceded Latest': ceded_latest, 'Ceded IBNR': ceded_ibnr,
        'Net Ult': net_ult, 'Net Latest': net_latest, 'Net IBNR': net_ibnr,
    })

result = pd.DataFrame(rows).set_index('AY')
print(f'30% Quota Share — Gross-minus-Ceded Reserve Projection:')
print(result.to_string())
print()
print(f'Total Gross IBNR:  {result["Gross IBNR"].sum():>10,.0f}')
print(f'Total Ceded IBNR:  {result["Ceded IBNR"].sum():>10,.0f}  (= {qs:.0%} of gross IBNR)')
print(f'Total Net IBNR:    {result["Net IBNR"].sum():>10,.0f}  (= {1-qs:.0%} of gross IBNR)')
print()
print('For quota share: Ceded IBNR = QS% × Gross IBNR. Simple proportional split.')

## Practice Problems

**Problem 1 — Salvage Estimation**

An auto physical damage insurer has the following history of gross paid losses and salvage received (both in $000s):

| AY | Gross Paid (Ult) | Salvage Received |
|---|---|---|
| 2020 | 15,200 | 1,140 |
| 2021 | 16,800 | 1,344 |
| 2022 | 18,100 | 1,448 |
| 2023 | 19,400 | 970 (immature) |

AY 2023 is immature — not all salvage has been collected. The projected gross ultimate for AY 2023 is $19,400. Using the mature AY average ratio, estimate: (a) the ultimate salvage for AY 2023, and (b) the salvage IBNR (future expected collections).

*Answer:*
- Mature ratios: 1,140/15,200 = 7.50%; 1,344/16,800 = 8.00%; 1,448/18,100 = 8.00%
- Selected ratio = (7.50% + 8.00% + 8.00%) / 3 = **7.83%**
- (a) Ultimate salvage AY 2023 = 19,400 × 7.83% = **$1,519**
- (b) Salvage IBNR = 1,519 − 970 = **$549**

---

**Problem 2 — XOL Treaty Terms**

An insurer has a per-occurrence XOL treaty: pays $750K excess of $250K. A large claim is reported at $1,200,000 gross. Calculate:
(a) The ceded portion
(b) The net (retained) portion

*Answer:*
- Retention = $250,000; Limit = $750,000; Reinsurer pays up to $750K above $250K
- (a) Ceded = min($1,200K, $250K + $750K) − min($1,200K, $250K) = min($1,200K, $1,000K) − $250K = $1,000K − $250K = **$750,000**
- (b) Net = $1,200,000 − $750,000 = **$450,000** ← wait, let's recalculate
- Actually: Net = min(Gross, Retention) = min($1,200K, $250K) = **$250,000**
- Check: $250K net + $750K ceded = $1,000K ≠ $1,200K → the loss exceeds the reinsurer's maximum payout!
- The insurer retains: $250K (retention) + $200K (above the limit) = **$450,000**
- Ceded = $750,000 (limited to the reinsurer's maximum)
- Check: $450K + $750K = $1,200K ✓

> **Key learning** — The XOL limit caps the reinsurer's payment. Losses above (Retention + Limit) are NOT covered by the reinsurer and revert to the primary insurer.

---

**Problem 3 — Gross vs. Net IBNR**

An insurer has a 40% quota share treaty. Its chain ladder analysis on the gross triangle produces gross IBNR of $12,000,000. The gross latest paid is $28,000,000. Ceded latest paid is $11,200,000 (= 40% × $28M).

Calculate: (a) ceded IBNR, (b) net IBNR, (c) net ultimate.

*Answer:*
- (a) Ceded IBNR = 40% × $12,000,000 = **$4,800,000**
- (b) Net IBNR = 60% × $12,000,000 = **$7,200,000**
- Net latest paid = $28,000,000 − $11,200,000 = $16,800,000
- Gross ultimate = $28,000,000 + $12,000,000 = $40,000,000
- (c) Net ultimate = 60% × $40,000,000 = **$24,000,000**
- Check: $16,800,000 + $7,200,000 = $24,000,000 ✓